# Early Prediction of Type 2 Diabetes - BEHRT Fine-Tuning

This notebook is responsible for the final predictive pipeline of the thesis. It trains and evaluates a customized **BEHRT (BERT for Electronic Health Records)** model on a binary sequence classification task.

###  Core Methodological Features:
* **Data Leakage Prevention:** We implemented strict temporal censoring. The model is forced to predict the future based solely on historical data, as all medical trajectories are explicitly truncated before the diagnosis index date.
* **Rigorous Class Balancing:** To prevent the model from defaulting to the majority class, the training dataset is dynamically balanced using random undersampling of the healthy cohort. The hold-out test set, however, is left entirely untouched to simulate real-world clinical environments.

## Step 1: Set up the Google Colab environment
We grant access to our files in Drive and install the necessary libraries.

In [6]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install the base BEHRT library (older version of HuggingFace)
!pip install pytorch_pretrained_bert


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2: Imports and Paths

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import pickle
from tqdm.notebook import tqdm  # Optimized progress bar for Colab
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

# --- ABSOLUTE GOOGLE DRIVE PATHS ---
TRAIN_PATH = '/content/drive/MyDrive/Colab Notebooks/BEHRT/0data/diabetes_train.parquet'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/BEHRT/0data/diabetes_test.parquet'
VOCAB_PATH = '/content/drive/MyDrive/Colab Notebooks/BEHRT/0data/vocab.pkl'
MODEL_WEIGHTS_PATH = '/content/drive/MyDrive/Colab Notebooks/BEHRT/saved_models/behrt_pretrain_model.pth'

# --- HYPERPARAMETERS ---
MAX_SEQ_LENGTH = 300
BATCH_SIZE = 64  # bumped from 32 -- pretraining had GPU headroom at this size, try it; drop back to 32 if you hit OOM
EPOCHS = 4
N_ITERATIONS = 10  # full retrain+eval repeated 10x, matching the LSTM's stability analysis methodology
BASE_SEED = 42     # each iteration uses BASE_SEED + iteration_index
LEARNING_RATE = 2e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--- Using processing device: {device} ---")


--- Using processing device: cuda ---


## Step 3: Robust Vocabulary Loading
The model needs to know how to translate a code like 'E11' (Diabetes) into its corresponding numerical ID.

In [8]:
print("Loading vocabulary...")
with open(VOCAB_PATH, 'rb') as f:
    vocab_obj = pickle.load(f)

if isinstance(vocab_obj, dict) and 'token2idx' in vocab_obj:
    word2idx = vocab_obj['token2idx']
    print("Successfully mapped 'token2idx' as the vocabulary.")
else:
    word2idx = vocab_obj
    print("Warning: 'token2idx' key not found, using raw object as vocabulary.")

VOCAB_SIZE = len(word2idx)
print(f"\nSUCCESS! Vocabulary loaded. Total unique codes in the hospital dataset: {VOCAB_SIZE}")

sep_id = word2idx.get('SEP', -1)
if sep_id == -1:
    print("WARNING: The 'SEP' token was not found in the vocabulary.")
else:
    print(f"'SEP' token correctly identified with ID: {sep_id}")

Loading vocabulary...
Successfully mapped 'token2idx' as the vocabulary.

SUCCESS! Vocabulary loaded. Total unique codes in the hospital dataset: 39268
'SEP' token correctly identified with ID: 3


## Step 4: DataLoader Construction
This transforms the `.parquet` tables into mathematical tensors that the GPU can process.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd
import duckdb

def read_parquet_safe(path):
    # Plain pd.read_parquet() throws ArrowNotImplementedError on parquet
    # files with large list-type columns (our 'code'/'age' sequences --
    # some patients have 200k+ tokens). DuckDB reads/converts these fine.
    con = duckdb.connect()
    return con.execute(f"SELECT * FROM read_parquet('{path}')").df()

class BEHRT_Diabetes_Dataset(Dataset):
    def __init__(self, filepath, word2idx, max_len=300):
        print(f"Reading parquet file: {filepath}...")
        self.df = read_parquet_safe(filepath)
        self.word2idx = word2idx
        self.max_len = max_len
        self.sep_id = self.word2idx.get('SEP', 0)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        codes = row['code']
        ages = row['age'] if 'age' in self.df.columns else [0] * len(codes)
        label = row['label']

        # Truncate to maximum length
        if len(codes) > self.max_len:
            codes = codes[-self.max_len:]
            ages = ages[-self.max_len:]

        # Convert codes to IDs
        # --- SAFETY BELT BOUNDARIES ---
        input_ids = [min(self.word2idx.get(c, 0), VOCAB_SIZE - 1) for c in codes]
        age_ids = [min(int(a) if str(a).isdigit() else 0, 1321) for a in ages]

        # Create Segment IDs (flip between 0 and 1 after each SEP)
        seg_ids = []
        current_seg = 0
        for code_id in input_ids:
            seg_ids.append(current_seg)
            if code_id == self.sep_id:
                current_seg = 1 - current_seg

        attention_mask = [1] * len(input_ids)

        # Padding
        padding_length = self.max_len - len(input_ids)
        if padding_length > 0:
            input_ids = input_ids + [0] * padding_length
            age_ids = age_ids + [0] * padding_length
            seg_ids = seg_ids + [0] * padding_length
            attention_mask = attention_mask + [0] * padding_length

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'age_ids': torch.tensor(age_ids, dtype=torch.long),
            'seg_ids': torch.tensor(seg_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.float)
        }

# Load the complete original datasets ONCE -- these are reused across all
# N_ITERATIONS below, only the undersampling indices change per iteration.
# Re-reading these parquet files 10 times would waste a lot of time for no
# benefit, since the underlying data never changes.
print("Loading full train/test datasets (shared across all iterations)...")
train_dataset = BEHRT_Diabetes_Dataset(TRAIN_PATH, word2idx, MAX_SEQ_LENGTH)
test_dataset = BEHRT_Diabetes_Dataset(TEST_PATH, word2idx, MAX_SEQ_LENGTH)

train_labels = train_dataset.df['label'].values
test_labels = test_dataset.df['label'].values

train_idx_class_1 = np.where(train_labels == 1)[0]
train_idx_class_0 = np.where(train_labels == 0)[0]
test_idx_class_1 = np.where(test_labels == 1)[0]
test_idx_class_0 = np.where(test_labels == 0)[0]

print(f"Train pool: {len(train_idx_class_1)} diabetic / {len(train_idx_class_0)} non-diabetic")
print(f"Test pool:  {len(test_idx_class_1)} diabetic / {len(test_idx_class_0)} non-diabetic")
print("(Balanced 1:1 undersampling of both is redone fresh, with a new seed, at the start of every iteration below.)")


NameError: name 'TRAIN_PATH' is not defined

## Step 5: Load the Pre-trained Model
We connect the weights from Phase 1 (MLM) to a new linear layer designed for binary classification.

In [ ]:
from pytorch_pretrained_bert import BertConfig, BertForSequenceClassification, BertModel

class BEHRTForSequenceClassification(nn.Module):
    def __init__(self, config, num_labels):
        super().__init__()
        self.bert = BertModel(config)
        # We adjust the size to 1322 so that it is the same as the pre-trained weights
        self.bert.embeddings.age_embeddings = nn.Embedding(1322, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(self, input_ids, age_ids, seg_ids, attention_mask):
        inputs_embeds = self.bert.embeddings.word_embeddings(input_ids)
        position_embeds = self.bert.embeddings.position_embeddings(
            torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        )
        token_type_embeds = self.bert.embeddings.token_type_embeddings(seg_ids)

        age_embeds = self.bert.embeddings.age_embeddings(age_ids)

        embeddings = inputs_embeds + position_embeds + token_type_embeds + age_embeds
        embeddings = self.bert.embeddings.LayerNorm(embeddings)
        embeddings = self.bert.embeddings.dropout(embeddings)

        encoder_output = self.bert.encoder(embeddings, attention_mask.unsqueeze(1).unsqueeze(2))

        if isinstance(encoder_output, tuple):
            sequence_output = encoder_output[0]
        else:
            sequence_output = encoder_output

        if isinstance(sequence_output, list):
            final_layer = sequence_output[-1]
        else:
            final_layer = sequence_output

        pooled_output = self.bert.pooler(final_layer)
        pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)

BERT_CONFIG = BertConfig(
    vocab_size_or_config_json_file=VOCAB_SIZE,
    hidden_size=288, num_hidden_layers=6, num_attention_heads=12,
    intermediate_size=512, hidden_act="gelu", hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1, max_position_embeddings=300  # must match pretraining max_seq_len=300, NOT 512
)

# Load the pretrained checkpoint ONCE -- reused (re-applied to a fresh
# model) at the start of every iteration below.
PRETRAINED_STATE_DICT = torch.load(MODEL_WEIGHTS_PATH, map_location=device)

def build_fresh_model():
    """Builds a brand new model instance, loaded from the SAME pretrained
    checkpoint every time. This is what makes the 10 iterations measure
    training-run-to-training-run variability (classifier head init, batch
    order, dropout, which majority-class patients got undersampled into
    train) instead of just continuing to fine-tune the previous iteration's
    already-fine-tuned weights."""
    m = BEHRTForSequenceClassification(BERT_CONFIG, num_labels=1)
    m.load_state_dict(PRETRAINED_STATE_DICT, strict=False)
    m.to(device)
    return m

print("Model builder ready (loads from pretrained checkpoint fresh on each call).")


## Step 6: Training (Fine-Tuning)
We fine-tune the model exclusively on the task of reading early historical context to predict incident diabetes.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Subset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from tqdm.notebook import tqdm

save_dir = '/content/drive/MyDrive/Colab Notebooks/BEHRT/saved_models'
results_dir = '/content/drive/MyDrive/Colab Notebooks/BEHRT/results'
os.makedirs(save_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print("========================================================")
print(f"STARTING {N_ITERATIONS}-ITERATION FINE-TUNING (DIABETES PREDICTION)")
print("========================================================")

iteration_metrics = []

for iteration in range(N_ITERATIONS):
    seed = BASE_SEED + iteration
    print("\n" + "="*60)
    print(f" ITERATION {iteration + 1}/{N_ITERATIONS}  (seed={seed})")
    print("="*60)

    # --- Fresh balanced 1:1 undersampling for THIS iteration (both train
    # and test), with its own seed -- different majority-class patients get
    # sampled in each iteration. ---
    rng = np.random.default_rng(seed)

    train_idx_class_0_downsampled = rng.choice(train_idx_class_0, size=len(train_idx_class_1), replace=False)
    balanced_train_indices = np.concatenate([train_idx_class_1, train_idx_class_0_downsampled])
    rng.shuffle(balanced_train_indices)
    balanced_train_dataset = Subset(train_dataset, balanced_train_indices)

    test_idx_class_0_downsampled = rng.choice(test_idx_class_0, size=len(test_idx_class_1), replace=False)
    balanced_test_indices = np.concatenate([test_idx_class_1, test_idx_class_0_downsampled])
    rng.shuffle(balanced_test_indices)
    balanced_test_dataset = Subset(test_dataset, balanced_test_indices)

    print(f"  Balanced train: {len(balanced_train_dataset)} patients | "
          f"Balanced test: {len(balanced_test_dataset)} patients")

    train_loader = DataLoader(balanced_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(balanced_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # --- Fresh model, reloaded from the pretrained checkpoint (NOT
    # continuing from the previous iteration's fine-tuned weights) ---
    torch.manual_seed(seed)
    model = build_fresh_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()

    # --- Train ---
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Iter {iteration + 1}/{N_ITERATIONS} Epoch {epoch + 1}/{EPOCHS}")

        for batch in progress_bar:
            b_input_ids = batch['input_ids'].to(device)
            b_age_ids = batch['age_ids'].to(device)
            b_seg_ids = batch['seg_ids'].to(device)
            b_mask = batch['attention_mask'].to(device)
            b_labels = batch['label'].to(device)

            model.zero_grad()
            outputs = model(b_input_ids, age_ids=b_age_ids, seg_ids=b_seg_ids, attention_mask=b_mask)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs
            logits = logits.squeeze(-1)

            loss = criterion(logits, b_labels)
            total_loss += loss.item()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_train_loss = total_loss / len(train_loader)
        print(f"  ---> Epoch {epoch + 1} Completed | Average Training Loss: {avg_train_loss:.4f}")

    checkpoint_path = os.path.join(save_dir, f'balanced_diabetes_model_iter_{iteration + 1}.pth')
    torch.save(model.state_dict(), checkpoint_path)
    print(f"  Checkpoint saved: {checkpoint_path}")

    # --- Evaluate on THIS iteration's balanced test set ---
    model.eval()
    all_probs, all_preds, all_labels_eval = [], [], []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Iter {iteration + 1}/{N_ITERATIONS} Evaluating"):
            b_input_ids = batch['input_ids'].to(device)
            b_age_ids = batch['age_ids'].to(device)
            b_seg_ids = batch['seg_ids'].to(device)
            b_mask = batch['attention_mask'].to(device)
            b_labels = batch['label'].cpu().numpy()

            outputs = model(b_input_ids, age_ids=b_age_ids, seg_ids=b_seg_ids, attention_mask=b_mask)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs
            logits = logits.squeeze(-1)

            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels_eval.extend(b_labels)

    cm = confusion_matrix(all_labels_eval, all_preds)
    TN, FP, FN, TP = cm.ravel()
    specificity = TN / (TN + FP) if (TN + FP) > 0 else np.nan

    iter_result = {
        'iteration': iteration + 1,
        'seed': seed,
        'roc_auc': roc_auc_score(all_labels_eval, all_probs),
        'f1': f1_score(all_labels_eval, all_preds),
        'precision': precision_score(all_labels_eval, all_preds),
        'recall': recall_score(all_labels_eval, all_preds),
        'specificity': specificity,
    }
    iteration_metrics.append(iter_result)

    print(f"  Iteration {iteration + 1} results: "
          f"AUC={iter_result['roc_auc']:.4f}  F1={iter_result['f1']:.4f}  "
          f"Prec={iter_result['precision']:.4f}  Rec={iter_result['recall']:.4f}  "
          f"Spec={iter_result['specificity']:.4f}")

    # Free GPU memory before the next iteration builds a new model
    del model, optimizer
    torch.cuda.empty_cache()

# ============================================================================
# Aggregate mean +/- std across all N_ITERATIONS -- this is the number to
# report in the thesis, matching the LSTM's stability-analysis methodology.
# ============================================================================
metrics_df = pd.DataFrame(iteration_metrics)
metrics_df.to_csv(os.path.join(results_dir, 'metrics_per_iteration.csv'), index=False)

summary_rows = []
for metric in ['roc_auc', 'f1', 'precision', 'recall', 'specificity']:
    summary_rows.append({
        'metric': metric,
        'mean': metrics_df[metric].mean(),
        'std': metrics_df[metric].std(),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(results_dir, 'metrics_summary_mean_std.csv'), index=False)

print("\n" + "="*60)
print(f" FINAL RESULTS ACROSS {N_ITERATIONS} ITERATIONS (mean +/- std) ")
print("="*60)
for _, row in summary_df.iterrows():
    print(f"  {row['metric']:12s}: {row['mean']:.4f} +/- {row['std']:.4f}")
print(f"\nPer-iteration results saved to: {os.path.join(results_dir, 'metrics_per_iteration.csv')}")
print(f"Summary (mean +/- std) saved to: {os.path.join(results_dir, 'metrics_summary_mean_std.csv')}")


## Step 7: The Final Exam (Test Set Evaluation)
We evaluate the model on patients it has never seen and calculate the scientific metrics for the thesis report.

In [ ]:
import matplotlib.pyplot as plt

# Bar chart with error bars (mean +/- std) across the N_ITERATIONS runs --
# ready to drop into the thesis. 'summary_df' and 'metrics_df' come from
# the loop cell above.
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    summary_df['metric'], summary_df['mean'], yerr=summary_df['std'],
    capsize=6, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title(f'Fine-tuning results across {N_ITERATIONS} iterations (mean \u00b1 std)')
for i, row in summary_df.iterrows():
    ax.text(i, row['mean'] + row['std'] + 0.02, f"{row['mean']:.3f}\u00b1{row['std']:.3f}",
            ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'metrics_summary_barplot.png'), dpi=300)
plt.show()

print(metrics_df.to_string(index=False))


## Step 8: The Confusion Matrix


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

results_dir = '/content/drive/MyDrive/Colab Notebooks/BEHRT/results'

# NOTE: 'cm' here is the confusion matrix from the LAST of the N_ITERATIONS runs only (loop variable left over from the cell above) -- it's a representative example, not an aggregate across iterations. The numbers to report are the mean +/- std from 'summary_df'.
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Healthy (0)', 'Predicted Diabetic (1)'],
            yticklabels=['True Healthy (0)', 'True Diabetic (1)'],
            annot_kws={"size": 14})

plt.title('Confusion Matrix - Type 2 Diabetes Prediction', fontsize=16, pad=15)
plt.xlabel('Model Prediction', fontsize=14, labelpad=10)
plt.ylabel('Actual Ground Truth', fontsize=14, labelpad=10)
plt.tight_layout()

# Save image directly to Google Drive
cm_path = os.path.join(results_dir, 'balanced_confusion_matrix.png')
plt.savefig(cm_path, dpi=300)
plt.show()

print(f" Confusion matrix successfully generated and saved at: {cm_path}")